# Flagging Vendor Invoices for Manual Review

**Objective:** Predict whether a vendor invoice should be flagged for manual approval based on abnormal cost, freight, or delivery patterns, in order to reduce financial risk, improve operational efficiency, and prioritize human review where it adds the most value.

- Manual invoice review is time-consuming and does not scale with transaction volume.
- Abnormal freight charges, pricing deviations, or delivery delays often indicate errors, disputes, or compliance risks.
- An automated flagging system enables finance teams to focus attention on high-risk invoices while allowing low-risk invoices to be processed automatically.

In [ ]:
import sqlite3
import pandas as pd
import seaborn as sns
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt

In [ ]:
conn=sqlite3.connect("/Users/shyamchauhan/Desktop/home/codes/project/vendor_invoice_intelligence_portal/data/inventory.db")
tables=pd.read_sql_query("select name from sqlite_master where type='table'",conn)

In [ ]:
for table in tables['name']:
    print('table name :',table)
    df=pd.read_sql_query(f"select * from {table} limit 5 ",conn)
    # display (df.dtypes)
    display (df)

In [ ]:
purchase_agg_data=pd.read_sql_query("""
select
p.PONumber,
count(distinct p.Brand) as total_brands,
sum(p.Quantity) as total_item_quantity,
sum(p.dollars) as total_item_dollars,
avg(julianday(p.ReceivingDate)-julianday(p.PODate)) as avg_receiving_delay

from purchases as p
group by p.PONumber

""",conn)

In [ ]:
purchase_agg_data

In [ ]:
df = pd.read_sql_query("""
-- Aggregate purchase data
WITH purchase_agg AS (
    SELECT
        p.PONumber,
        COUNT(DISTINCT p.Brand) AS total_brands,
        SUM(p.Quantity) AS total_item_quantity,
        SUM(p.Dollars) AS total_item_dollars,
        AVG(julianday(p.ReceivingDate) - julianday(p.PODate)) AS avg_receiving_delay
    FROM purchases p
    GROUP BY p.PONumber
)

-- Join with vendor invoices
SELECT
    vi.PONumber,
    vi.Quantity AS invoice_quantity,
    vi.Dollars AS invoice_dollars,
    vi.Freight,
    (julianday(vi.InvoiceDate) - julianday(vi.PODate)) AS days_po_to_invoice,
    (julianday(vi.PayDate) - julianday(vi.InvoiceDate)) AS days_to_pay,
    pa.total_brands,
    pa.total_item_quantity,
    pa.total_item_dollars,
    pa.avg_receiving_delay
FROM vendor_invoice vi
LEFT JOIN purchase_agg pa
    ON vi.PONumber = pa.PONumber
""", conn)

In [ ]:
print(df.isnull().sum())
df.dtypes

In [ ]:
df.head(5)

In [ ]:
# does invoice require manual approval or not
def create_invoice_risk_label(row):
    # Invoice total mismatch with item-level total
    # require manual approval if total items cost does not match with invoice invoice cost
    if abs(row["invoice_dollars"] - row["total_item_dollars"]) > 5:
        return 1

    # Abnormally high receiving delay
    if row["avg_receiving_delay"] > 10:
        return 1
    return 0

df["flag_invoice"] = df.apply(create_invoice_risk_label, axis=1)
df["flag_invoice"].value_counts()

In [ ]:
plt.figure(figsize=(20,10))
sns.heatmap(df.iloc[:,1:-1].corr(),annot=True)
plt.show()

In [ ]:
flagged=df[df['flag_invoice']==1]
normal=df[df['flag_invoice']==0]

In [ ]:
# t test is used when mean and sd is unknown and we need to check if there is signnificant difference
significant_features=[]
non_significant_features=[]
results=[]

In [ ]:
metrics = [
    'invoice_quantity',
    'invoice_dollars',
    'Freight',
    'days_po_to_invoice',
    'days_to_pay',
    'total_brands',
    'total_item_quantity',
    'total_item_dollars',
    'avg_receiving_delay'
]

In [ ]:
'''
significant difference means observed difference is unlikely to have happened by chance ... likely reflects real difference between groups.
p_value < 0.05 ==> there is significant difference
p_value > 0.05 ==> no significant difference
'''
for metric in metrics:
    flagged_mean=flagged[metric].mean()
    normal_mean=normal[metric].mean()

    t_stats,p_value=ttest_ind(
        flagged[metric].dropna(),
        normal[metric].dropna(),
        equal_var=False
    )
    if p_value<0.05:
        significant_features.append(metric)
        results.append({
            "metric":metric,
            "flagged_mean":float(flagged_mean.round(2)),
            "normal_mean":float(normal_mean.round(2)),
            "p_value":float(p_value.round(3))
        })
    else:
        non_significant_features.append(metric)
        print(metric)
        print({
            "metric":metric,
            "flagged_mean":float(flagged_mean.round(2)),
            "normal_mean":float(normal_mean.round(2)),
            "p_value":float(p_value.round(3))
        })

In [ ]:
non_significant_features

In [ ]:
significant_features

In [ ]:
results

In [ ]:
X = df[['invoice_quantity', 'invoice_dollars', 'Freight',
        'total_brands', 'total_item_quantity',
        'days_po_to_invoice', 'total_item_dollars']]

y = df['flag_invoice']

In [ ]:
X.describe().round()

In [ ]:
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
scaler=MinMaxScaler()
x_trained_scaled=scaler.fit_transform(X_train)
x_test_scaled=scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Logistic Regression
model1 = LogisticRegression(random_state=42)
model1.fit(x_trained_scaled, y_train)

# Decision Tree
model2 = DecisionTreeClassifier(random_state=42)
model2.fit(x_trained_scaled, y_train)

# Random Forest
model3 = RandomForestClassifier(random_state=42)
model3.fit(x_trained_scaled, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"\n{model_name}")
    print(f"Accuracy: {acc:.2f}") 
    print(classification_report(y_test, y_pred))

In [ ]:
evaluate_model(model1,x_test_scaled,y_test,'Logistic regression')
evaluate_model(model2, x_test_scaled, y_test, 'Decision Tree Classifier')
evaluate_model(model3, x_test_scaled, y_test, 'Random Forest Classifier')

In [ ]:
model3.feature_importances_

In [ ]:
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model3.feature_importances_
}).sort_values(by="importance", ascending=False)

feature_importance

In [ ]:
X = df[['invoice_quantity', 'invoice_dollars', 'Freight', 'total_item_quantity', 'total_item_dollars']]
y = df['flag_invoice']

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model3 = RandomForestClassifier(random_state=42)
model3.fit(X_train_scaled, y_train)
evaluate_model(model3, X_test_scaled, y_test, 'Random Forest Classifier')

In [ ]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 1, 2, 3],
    "min_samples_split": [2, 3, 5],
    "min_samples_leaf": [1, 2, 5],
    "criterion": ['gini', 'entropy']
}

In [ ]:
from sklearn.metrics import make_scorer, f1_score
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

scorer = make_scorer(f1_score)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring=scorer,
    cv=5,
    verbose=2,
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

evaluate_model(grid_search, X_test_scaled, y_test, 'Random Forest Classifier')

In [ ]:
from sklearn.metrics import confusion_matrix
confusion_matrix(grid_search.predict(X_test_scaled),y_test)

In [ ]:
confusion_matrix(model3.predict(X_test_scaled),y_test)

In [ ]:
grid_search.best_params_